# Create DeLong Table

Compares all preop and combined models within each group and with the other group

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Pair-wise AUC comparison with a correct implementation of DeLong’s test
(Modified for improved formatting and output)
"""

import os
import html
import numpy as np
import pandas as pd
import scipy.stats
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
from IPython.display import display, HTML

# =============================================================================
# SCRIPT CONFIGURATION
# =============================================================================
RESULTS_DIR = '/home/server/Projects/data/AKI/results/'
OUTPUT_CSV_FILE = 'delong_comparison_results.csv'

MODELS_TO_EXCLUDE = [
    'preop_ASA Rule',
    'combined_ASA Rule',
    'combined_Hybrid (MLP + LSTM)'
]

# --- MODIFICATION: Updated model names for consistency ---
model_translate = {
    'log_reg': 'LR', 'xgb': 'GBT',
    'svm': 'SVM', # MODIFIED: Changed key to 'svm'
    'mlp': 'MLP', 'rf': 'RF',
    'knn': 'KNN', 'asa_rule': 'ASA Rule', 'autogluon': 'AutoGluon',
    'lstm': 'LSTM', 'hybrid': 'MLP+LSTM'
}

model_base_map = {
    'LSTM': 'base_54k', 
    'MLP+LSTM': 'base_54k'
}
DEFAULT_BASE = 'base'

# =============================================================================
# FAST & MEMORY-SAFE DELONG (no big matrices)
# =============================================================================
def _structural_components(y_true: np.ndarray,
                           scores:   np.ndarray):
    """
    Return (auc, v10, v01) without building an n_pos × n_neg table.

    v10[i]  = mean_j θ( s_pos[i] , s_neg[j] )
    v01[j]  = mean_i θ( s_pos[i] , s_neg[j] ),      θ = 1, 0.5, 0
    """
    y_true  = y_true.astype(int).ravel()
    scores  = scores.ravel()
    pos     = scores[y_true == 1]
    neg     = scores[y_true == 0]

    n_pos   = pos.size
    n_neg   = neg.size
    if n_pos == 0 or n_neg == 0:
        raise ValueError("Need at least one positive and one negative sample.")

    # Pre-sort once for binary-search look-ups
    neg_sorted = np.sort(neg)
    pos_sorted = np.sort(pos)

    # --- V10: for every positive score, compare with all negatives -----------
    left  = np.searchsorted(neg_sorted, pos, side='left')   # #neg < s_pos
    right = np.searchsorted(neg_sorted, pos, side='right')  # #neg ≤ s_pos
    ties  = right - left
    v10   = (left + 0.5 * ties) / n_neg                     # shape (n_pos,)

    # --- V01: for every negative score, compare with all positives -----------
    left  = np.searchsorted(pos_sorted, neg, side='left')   # #pos < s_neg
    right = np.searchsorted(pos_sorted, neg, side='right')  # #pos ≤ s_neg
    ties  = right - left
    greater = n_pos - right
    v01   = (greater + 0.5 * ties) / n_pos                  # shape (n_neg,)

    auc = v10.mean()                                        # identical to roc_auc_score
    return auc, v10, v01


def delong_test(y_true:  np.ndarray,
                scores1: np.ndarray,
                scores2: np.ndarray):
    """
    DeLong’s test for correlated AUCs (two-sided).
    Returns (auc1, auc2, p_value).
    Memory complexity: O(n) ;  never builds n_pos × n_neg arrays.
    """
    auc1, v10_1, v01_1 = _structural_components(y_true, scores1)
    auc2, v10_2, v01_2 = _structural_components(y_true, scores2)

    n_pos = v10_1.size
    n_neg = v01_1.size

    s10 = np.cov(np.vstack([v10_1, v10_2]), ddof=1)
    s01 = np.cov(np.vstack([v01_1, v01_2]), ddof=1)

    var_auc_diff = (s10[0, 0] + s10[1, 1] - 2 * s10[0, 1]) / n_pos + \
                   (s01[0, 0] + s01[1, 1] - 2 * s01[0, 1]) / n_neg

    if var_auc_diff <= 0 or not np.isfinite(var_auc_diff):
        return auc1, auc2, 1.0        # numerical safeguard

    z = (auc1 - auc2) / np.sqrt(var_auc_diff)
    p = 2 * (1 - norm.cdf(abs(z)))
    return auc1, auc2, p

# =============================================================================
# DATA PREPARATION AND ANALYSIS FUNCTIONS
# =============================================================================
def prepare_and_flatten_data(data_sources):
    """Loads and flattens predictions for each requested source."""
    prepared_data = {}
    print("--- Preparing and Loading Data ---")
    for source_name in data_sources:
        file_path = os.path.join(RESULTS_DIR, f"tabular_{source_name}_test.pkl")
        try:
            df = pd.read_pickle(file_path)
            print(f"Loaded: {file_path}")
        except FileNotFoundError:
            print(f"ERROR: {file_path} not found → skipped.")
            continue

        # Ground-truth arrays (“base” models) ---------------------------
        base_models_data = {}
        base_df = df[df['model_name'].str.contains('base', na=False)]
        for _, row in base_df.iterrows():
            base_models_data[row['model_name']] = np.concatenate(row['y_pred_binary'])

        # Predictive models --------------------------------------------
        models_df = df[~df['model_name'].str.contains('base', na=False)]
        for _, row in models_df.iterrows():
            model_key = row['model_name']
            if model_key not in model_translate:
                continue

            model_name_trans = model_translate[model_key]
            full_name = f"{source_name}_{model_name_trans}"

            base_to_use = model_base_map.get(model_name_trans, DEFAULT_BASE)
            if base_to_use not in base_models_data:
                print(f"WARNING: base '{base_to_use}' missing for {full_name} → skipped.")
                continue

            y_true_flat = base_models_data[base_to_use]
            y_prob_flat = np.concatenate(row['y_prob'])

            if len(y_true_flat) != len(y_prob_flat):
                print(f"WARNING: length mismatch for {full_name} → skipped.")
                continue

            prepared_data[full_name] = (y_true_flat, y_prob_flat)

    print(f"Prepared data for {len(prepared_data)} models.")
    return prepared_data

def run_comprehensive_delong_tests(prepared_data):
    """Runs DeLong’s test for every model pair that shares identical ground truth."""
    print("--- Running Pairwise DeLong Tests ---")
    all_models = sorted(m for m in prepared_data if m not in MODELS_TO_EXCLUDE)

    if not all_models:
        print("ERROR: No models left to compare.")
        return None

    grid = pd.DataFrame(index=all_models, columns=all_models, dtype=float)
    tot = len(all_models) ** 2
    done = 0

    for m1 in all_models:
        for m2 in all_models:
            done += 1
            print(f"[{done}/{tot}] {m1} vs {m2}")
            if m1 == m2:
                grid.loc[m1, m2] = np.nan
                continue

            y1, p1 = prepared_data[m1]
            y2, p2 = prepared_data[m2]

            if not np.array_equal(y1, y2):
                print("   → GT mismatch, set N/A.")
                grid.loc[m1, m2] = "N/A"
                continue

            _, _, p = delong_test(y1, p1, p2)
            grid.loc[m1, m2] = p

    return grid

def format_and_save_results(p_value_grid):
    """Pretty-print, write CSV, and show a styled HTML table in Jupyter."""
    print("--- Formatting Results ---")

    # --- MODIFICATION: Rename models for concise table display ---
    renamed_grid = p_value_grid.copy()
    new_names = [name.replace('preop_', 'p_').replace('intraop_','i_').replace('combined_', 'c_') for name in renamed_grid.columns]
    renamed_grid.columns = new_names
    renamed_grid.index = new_names

    # Formatting function for p-values with significance stars
    def fmt(p):
        if pd.isna(p): return "nan"
        if isinstance(p, str): return p
        if p < 1e-4: return "<0.0001***"
        if p < 1e-3: return f"{p:.4f}***"
        if p < 1e-2: return f"{p:.4f}**"
        if p < 5e-2: return f"{p:.4f}*"
        return f"{p:.4f}"

    text_grid = renamed_grid.map(fmt)
    print("\nDeLong Test Results (p_ = preop, i_ = intraop, c_ = combined):\n")
    print(text_grid.to_string())

    text_grid.to_csv(OUTPUT_CSV_FILE)
    print(f"\nCSV written to {OUTPUT_CSV_FILE}")

    # --- MODIFICATION: Jupyter-friendly styled table for Google Docs ---
    # Style significant p-values with bold text AND a background color
    def highlight_significant(p):
        is_sig = isinstance(p, (float, int)) and p < 0.05
        # NEW: Return a string with both CSS properties if significant
        return 'font-weight: bold; background-color: #d4edda;' if is_sig else ''

    styled = (
        renamed_grid.style
        .map(highlight_significant)
        .format(fmt)
        .set_caption("Pairwise DeLong Test P-Values")
        .set_table_styles([
            # Set font for the entire table for better copy-paste compatibility
            {'selector': '', 'props': [('font-family', '"Times New Roman", Times, serif'),
                                       ('border-collapse', 'collapse')]},
            # Style for header cells
            {'selector': 'th', 'props': [('text-align', 'left'), ('font-weight', 'bold'),
                                         ('padding', '6px 8px'), ('border', '1px solid black')]},
            # Style for data cells
            {'selector': 'td', 'props': [('text-align', 'left'), ('padding', '6px 8px'),
                                         ('border', '1px solid black')]},
            # Style for the caption
            {'selector': 'caption', 'props': [('caption-side', 'top'),
                                              ('font-size', '1.2em'),
                                              ('font-weight', 'bold'),
                                              ('margin-bottom', '10px'),
                                              ('color', 'black')]}
        ])
    )

    try:
        # --- MODIFICATION: Create collapsible widget to show raw HTML ---
        table_html = styled.to_html()
        escaped_html = html.escape(table_html)

        widget_html = f"""
        <details style="margin-top: 20px; font-family: monospace;">
            <summary style="cursor: pointer; font-family: sans-serif; color: #0073e6;">
                Click to view raw HTML for table
            </summary>
            <pre style="background-color:#f4f4f4; border:1px solid #ddd; padding:10px; border-radius:5px; white-space: pre-wrap; word-wrap: break-word;"><code>{escaped_html}</code></pre>
        </details>
        """

        # Combine the visible table and the hidden widget for final output
        final_output = table_html + widget_html
        display(HTML(final_output))

    except NameError:
        # display() is not available outside of a Jupyter/IPython environment.
        pass

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    # Now includes 'intraop'
    data_to_compare = prepare_and_flatten_data(['preop', 'intraop', 'combined'])

    if data_to_compare:
        p_grid = run_comprehensive_delong_tests(data_to_compare)
        if p_grid is not None:
            format_and_save_results(p_grid)
    else:
        print("No data prepared – exiting.")


--- Preparing and Loading Data ---
Loaded: /home/server/Projects/data/AKI/results/tabular_preop_test.pkl
Loaded: /home/server/Projects/data/AKI/results/tabular_intraop_test.pkl
Loaded: /home/server/Projects/data/AKI/results/tabular_combined_test.pkl
Prepared data for 25 models.
--- Running Pairwise DeLong Tests ---
[1/529] combined_AutoGluon vs combined_AutoGluon
[2/529] combined_AutoGluon vs combined_GBT
[3/529] combined_AutoGluon vs combined_KNN
[4/529] combined_AutoGluon vs combined_LR
[5/529] combined_AutoGluon vs combined_MLP
[6/529] combined_AutoGluon vs combined_MLP+LSTM
   → GT mismatch, set N/A.
[7/529] combined_AutoGluon vs combined_RF


/tmp/ipykernel_459170/4149129743.py:188: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grid.loc[m1, m2] = "N/A"


[8/529] combined_AutoGluon vs combined_SVM
[9/529] combined_AutoGluon vs intraop_AutoGluon
[10/529] combined_AutoGluon vs intraop_GBT
[11/529] combined_AutoGluon vs intraop_KNN
[12/529] combined_AutoGluon vs intraop_LR
[13/529] combined_AutoGluon vs intraop_LSTM
   → GT mismatch, set N/A.
[14/529] combined_AutoGluon vs intraop_MLP


/tmp/ipykernel_459170/4149129743.py:188: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grid.loc[m1, m2] = "N/A"


[15/529] combined_AutoGluon vs intraop_RF
[16/529] combined_AutoGluon vs intraop_SVM
[17/529] combined_AutoGluon vs preop_AutoGluon
[18/529] combined_AutoGluon vs preop_GBT
[19/529] combined_AutoGluon vs preop_KNN
[20/529] combined_AutoGluon vs preop_LR
[21/529] combined_AutoGluon vs preop_MLP
[22/529] combined_AutoGluon vs preop_RF
[23/529] combined_AutoGluon vs preop_SVM
[24/529] combined_GBT vs combined_AutoGluon
[25/529] combined_GBT vs combined_GBT
[26/529] combined_GBT vs combined_KNN
[27/529] combined_GBT vs combined_LR
[28/529] combined_GBT vs combined_MLP
[29/529] combined_GBT vs combined_MLP+LSTM
   → GT mismatch, set N/A.
[30/529] combined_GBT vs combined_RF
[31/529] combined_GBT vs combined_SVM
[32/529] combined_GBT vs intraop_AutoGluon
[33/529] combined_GBT vs intraop_GBT
[34/529] combined_GBT vs intraop_KNN
[35/529] combined_GBT vs intraop_LR
[36/529] combined_GBT vs intraop_LSTM
   → GT mismatch, set N/A.
[37/529] combined_GBT vs intraop_MLP
[38/529] combined_GBT vs intr

/tmp/ipykernel_459170/4149129743.py:188: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grid.loc[m1, m2] = "N/A"
/tmp/ipykernel_459170/4149129743.py:188: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grid.loc[m1, m2] = "N/A"
/tmp/ipykernel_459170/4149129743.py:188: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grid.loc[m1, m2] = "N/A"
/tmp/ipykernel_459170/4149129743.py:188: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an e

[129/529] combined_MLP+LSTM vs intraop_MLP
   → GT mismatch, set N/A.
[130/529] combined_MLP+LSTM vs intraop_RF
   → GT mismatch, set N/A.
[131/529] combined_MLP+LSTM vs intraop_SVM
   → GT mismatch, set N/A.
[132/529] combined_MLP+LSTM vs preop_AutoGluon
   → GT mismatch, set N/A.
[133/529] combined_MLP+LSTM vs preop_GBT
   → GT mismatch, set N/A.
[134/529] combined_MLP+LSTM vs preop_KNN
   → GT mismatch, set N/A.
[135/529] combined_MLP+LSTM vs preop_LR
   → GT mismatch, set N/A.
[136/529] combined_MLP+LSTM vs preop_MLP
   → GT mismatch, set N/A.
[137/529] combined_MLP+LSTM vs preop_RF
   → GT mismatch, set N/A.
[138/529] combined_MLP+LSTM vs preop_SVM
   → GT mismatch, set N/A.
[139/529] combined_RF vs combined_AutoGluon
[140/529] combined_RF vs combined_GBT
[141/529] combined_RF vs combined_KNN
[142/529] combined_RF vs combined_LR
[143/529] combined_RF vs combined_MLP
[144/529] combined_RF vs combined_MLP+LSTM
   → GT mismatch, set N/A.
[145/529] combined_RF vs combined_RF
[146/529]

,c_AutoGluon,c_GBT,c_KNN,c_LR,c_MLP,c_MLP+LSTM,c_RF,c_SVM,i_AutoGluon,i_GBT,i_KNN,i_LR,i_LSTM,i_MLP,i_RF,i_SVM,p_AutoGluon,p_GBT,p_KNN,p_LR,p_MLP,p_RF,p_SVM
c_AutoGluon,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,N/A,0.0834,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.5669,<0.0001***,<0.0001***,<0.0001***,0.3820,<0.0001***
c_GBT,<0.0001***,nan,<0.0001***,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
c_KNN,<0.0001***,<0.0001***,nan,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
c_LR,<0.0001***,<0.0001***,<0.0001***,nan,0.4340,N/A,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.2469,<0.0001***,<0.0001***
c_MLP,<0.0001***,<0.0001***,<0.0001***,0.4340,nan,N/A,<0.0001***,0.8059,<0.0001***,<0.0001***,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,<0.0001***,0.0003***,<0.0001***,<0.0001***,<0.0001***,0.6381,<0.0001***,<0.0001***
c_MLP+LSTM,N/A,N/A,N/A,N/A,N/A,nan,N/A,N/A,N/A,N/A,N/A,N/A,<0.0001***,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
c_RF,0.0834,<0.0001***,<0.0001***,<0.0001***,<0.0001***,N/A,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.3963,<0.0001***,<0.0001***,<0.0001***,0.3863,<0.0001***
c_SVM,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.8059,N/A,<0.0001***,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,<0.0001***,0.0002***,<0.0001***,<0.0001***,<0.0001***,0.7720,<0.0001***,<0.0001***
i_AutoGluon,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,nan,<0.0001***,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
i_GBT,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,<0.0001***,nan,<0.0001***,<0.0001***,N/A,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***


## With FDR Correction

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Pair-wise AUC comparison with DeLong’s test and Benjamini-Hochberg FDR Correction.
"""

import os
import html
import numpy as np
import pandas as pd
import scipy.stats
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests
from IPython.display import display, HTML

# =============================================================================
# SCRIPT CONFIGURATION
# =============================================================================
RESULTS_DIR = '/home/server/Projects/data/AKI/results/'
OUTPUT_CSV_FILE = 'delong_fdr_corrected_results.csv' # New output file

MODELS_TO_EXCLUDE = [
    'preop_ASA Rule',
    'combined_ASA Rule',
    'combined_Hybrid (MLP + LSTM)'
]

# Model names consistent with other scripts
model_translate = {
    'log_reg': 'LR', 'xgb': 'GBT',
    'svm': 'SVM', 
    'mlp': 'MLP', 'rf': 'RF',
    'knn': 'KNN', 'asa_rule': 'ASA Rule', 'autogluon': 'AutoGluon',
    'lstm': 'LSTM', 'hybrid': 'MLP+LSTM'
}

model_base_map = {
    'LSTM': 'base_54k', 
    'MLP+LSTM': 'base_54k'
}
DEFAULT_BASE = 'base'

# =============================================================================
# DELONG'S TEST IMPLEMENTATION (UNCHANGED)
# =============================================================================
def _structural_components(y_true: np.ndarray, scores: np.ndarray):
    y_true  = y_true.astype(int).ravel()
    scores  = scores.ravel()
    pos     = scores[y_true == 1]
    neg     = scores[y_true == 0]
    n_pos, n_neg = pos.size, neg.size
    if n_pos == 0 or n_neg == 0:
        raise ValueError("Need at least one positive and one negative sample.")
    neg_sorted = np.sort(neg)
    pos_sorted = np.sort(pos)
    left  = np.searchsorted(neg_sorted, pos, side='left')
    right = np.searchsorted(neg_sorted, pos, side='right')
    v10 = (left + 0.5 * (right - left)) / n_neg
    left  = np.searchsorted(pos_sorted, neg, side='left')
    right = np.searchsorted(pos_sorted, neg, side='right')
    v01 = (n_pos - right + 0.5 * (right - left)) / n_pos
    return v10.mean(), v10, v01

def delong_test(y_true: np.ndarray, scores1: np.ndarray, scores2: np.ndarray):
    auc1, v10_1, v01_1 = _structural_components(y_true, scores1)
    auc2, v10_2, v01_2 = _structural_components(y_true, scores2)
    n_pos, n_neg = v10_1.size, v01_1.size
    s10 = np.cov(np.vstack([v10_1, v10_2]), ddof=1)
    s01 = np.cov(np.vstack([v01_1, v01_2]), ddof=1)
    var_auc_diff = (s10[0,0] + s10[1,1] - 2*s10[0,1])/n_pos + (s01[0,0] + s01[1,1] - 2*s01[0,1])/n_neg
    if var_auc_diff <= 0 or not np.isfinite(var_auc_diff): return auc1, auc2, 1.0
    z = (auc1 - auc2) / np.sqrt(var_auc_diff)
    p = 2 * (1 - norm.cdf(abs(z)))
    return auc1, auc2, p

# =============================================================================
# DATA PREPARATION AND ANALYSIS FUNCTIONS
# =============================================================================
def prepare_and_flatten_data(data_sources):
    """Loads and flattens predictions for each requested source."""
    prepared_data = {}
    print("--- Preparing and Loading Data ---")
    for source_name in data_sources:
        file_path = os.path.join(RESULTS_DIR, f"tabular_{source_name}_test.pkl")
        try:
            df = pd.read_pickle(file_path)
            print(f"Loaded: {file_path}")
        except FileNotFoundError:
            print(f"ERROR: {file_path} not found → skipped.")
            continue
        
        base_models_data = {row['model_name']: np.concatenate(row['y_pred_binary']) 
                            for _, row in df[df['model_name'].str.contains('base', na=False)].iterrows()}

        models_df = df[~df['model_name'].str.contains('base', na=False)]
        for _, row in models_df.iterrows():
            model_key = row['model_name']
            if model_key not in model_translate: continue
            
            model_name_trans = model_translate[model_key]
            full_name = f"{source_name}_{model_name_trans}"
            base_to_use = model_base_map.get(model_name_trans, DEFAULT_BASE)

            if base_to_use not in base_models_data:
                print(f"WARNING: base '{base_to_use}' missing for {full_name} → skipped.")
                continue

            y_true_flat = base_models_data[base_to_use]
            y_prob_flat = np.concatenate(row['y_prob'])

            if len(y_true_flat) != len(y_prob_flat):
                print(f"WARNING: length mismatch for {full_name} → skipped.")
                continue
            prepared_data[full_name] = (y_true_flat, y_prob_flat)

    print(f"Prepared data for {len(prepared_data)} models.")
    return prepared_data

def run_comprehensive_delong_tests(prepared_data):
    """Runs DeLong’s test for every model pair that shares identical ground truth."""
    print("--- Running Pairwise DeLong Tests ---")
    all_models = sorted(m for m in prepared_data if m not in MODELS_TO_EXCLUDE)
    if not all_models:
        print("ERROR: No models left to compare."); return None
    
    grid = pd.DataFrame(index=all_models, columns=all_models, dtype=float)
    
    for i, m1 in enumerate(all_models):
        for j, m2 in enumerate(all_models):
            if i >= j: continue # Only compute upper triangle
            
            y1, p1 = prepared_data[m1]
            y2, p2 = prepared_data[m2]

            if not np.array_equal(y1, y2):
                grid.loc[m1, m2] = np.nan # Mark as NaN if ground truth differs
                continue

            _, _, p = delong_test(y1, p1, p2)
            grid.loc[m1, m2] = p
            
    return grid

def apply_fdr_correction(p_value_grid):
    """
    Applies Benjamini-Hochberg FDR correction to a grid of p-values.
    """
    print("--- Applying Benjamini-Hochberg FDR Correction ---")
    
    # Extract the p-values from the upper triangle of the grid, ignoring NaNs
    p_values_flat = p_value_grid.values[np.triu_indices_from(p_value_grid.values, k=1)]
    p_values_flat = p_values_flat[~np.isnan(p_values_flat)]

    if len(p_values_flat) == 0:
        print("No p-values to correct.")
        return p_value_grid

    # Apply the Benjamini-Hochberg correction
    _, p_corrected, _, _ = multipletests(p_values_flat, alpha=0.05, method='fdr_bh')
    
    # Create a new grid to store the corrected p-values
    corrected_grid = p_value_grid.copy()
    
    # Place the corrected p-values back into the new grid
    p_corr_iter = 0
    for i in range(len(corrected_grid.index)):
        for j in range(i + 1, len(corrected_grid.columns)):
            if not np.isnan(corrected_grid.iloc[i, j]):
                corrected_grid.iloc[i, j] = p_corrected[p_corr_iter]
                p_corr_iter += 1
    
    # Mirror the upper triangle to the lower triangle
    corrected_grid.values[np.tril_indices_from(corrected_grid.values, k=-1)] = corrected_grid.values.T[np.tril_indices_from(corrected_grid.values, k=-1)]
    
    return corrected_grid

def format_and_save_results(p_value_grid, corrected=False):
    """Pretty-print, write CSV, and show a styled HTML table in Jupyter."""
    print("--- Formatting Results ---")
    
    caption_text = "Pairwise DeLong Test P-Values (FDR Corrected)" if corrected else "Pairwise DeLong Test P-Values"
    
    renamed_grid = p_value_grid.copy()
    new_names = [name.replace('preop_', 'p_').replace('intraop_','i_').replace('combined_', 'c_') for name in renamed_grid.columns]
    renamed_grid.columns = new_names
    renamed_grid.index = new_names

    def fmt(p):
        if pd.isna(p) or p > 1: return "---"
        if isinstance(p, str): return p
        if p < 1e-4: return "<0.0001***"
        if p < 1e-3: return f"{p:.4f}***"
        if p < 1e-2: return f"{p:.4f}**"
        if p < 5e-2: return f"{p:.4f}*"
        return f"{p:.4f}"

    text_grid = renamed_grid.map(fmt)
    print(f"\n{caption_text} (p_ = preop, i_ = intraop, c_ = combined):\n")
    print(text_grid.to_string())

    text_grid.to_csv(OUTPUT_CSV_FILE)
    print(f"\nCSV written to {OUTPUT_CSV_FILE}")

    def highlight_significant(p):
        is_sig = isinstance(p, (float, int)) and p < 0.05
        return 'font-weight: bold; background-color: #d4edda;' if is_sig else ''

    styled = (
        renamed_grid.style.map(highlight_significant).format(fmt)
        .set_caption(caption_text)
        .set_table_styles([
            {'selector': '', 'props': [('font-family', '"Times New Roman", Times, serif'), ('border-collapse', 'collapse')]},
            {'selector': 'th', 'props': [('text-align', 'left'), ('font-weight', 'bold'), ('padding', '6px 8px'), ('border', '1px solid black')]},
            {'selector': 'td', 'props': [('text-align', 'left'), ('padding', '6px 8px'), ('border', '1px solid black')]},
            {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-size', '1.2em'), ('font-weight', 'bold'), ('margin-bottom', '10px'), ('color', 'black')]}
        ])
    )

    try:
        table_html = styled.to_html()
        escaped_html = html.escape(table_html)
        widget_html = f"""
        <details style="margin-top: 20px; font-family: monospace;">
            <summary style="cursor: pointer; font-family: sans-serif; color: #0073e6;">Click to view raw HTML for table</summary>
            <pre style="background-color:#f4f4f4; border:1px solid #ddd; padding:10px; border-radius:5px; white-space: pre-wrap; word-wrap: break-word;"><code>{escaped_html}</code></pre>
        </details>
        """
        display(HTML(table_html + widget_html))
    except NameError:
        pass

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    data_to_compare = prepare_and_flatten_data(['preop', 'intraop', 'combined'])

    if data_to_compare:
        raw_p_grid = run_comprehensive_delong_tests(data_to_compare)
        if raw_p_grid is not None:
            corrected_p_grid = apply_fdr_correction(raw_p_grid)
            format_and_save_results(corrected_p_grid, corrected=True)
    else:
        print("No data prepared – exiting.")


--- Preparing and Loading Data ---
Loaded: /home/server/Projects/data/AKI/results/tabular_preop_test.pkl
Loaded: /home/server/Projects/data/AKI/results/tabular_intraop_test.pkl
Loaded: /home/server/Projects/data/AKI/results/tabular_combined_test.pkl
Prepared data for 25 models.
--- Running Pairwise DeLong Tests ---
--- Applying Benjamini-Hochberg FDR Correction ---
--- Formatting Results ---

Pairwise DeLong Test P-Values (FDR Corrected) (p_ = preop, i_ = intraop, c_ = combined):

            c_AutoGluon       c_GBT       c_KNN        c_LR       c_MLP  c_MLP+LSTM        c_RF       c_SVM i_AutoGluon       i_GBT       i_KNN        i_LR      i_LSTM       i_MLP        i_RF       i_SVM p_AutoGluon       p_GBT       p_KNN        p_LR       p_MLP        p_RF       p_SVM
c_AutoGluon         ---  <0.0001***  <0.0001***  <0.0001***  <0.0001***         ---      0.0875  <0.0001***  <0.0001***  <0.0001***  <0.0001***  <0.0001***         ---  <0.0001***  <0.0001***  <0.0001***  <0.0001***      0.577

,c_AutoGluon,c_GBT,c_KNN,c_LR,c_MLP,c_MLP+LSTM,c_RF,c_SVM,i_AutoGluon,i_GBT,i_KNN,i_LR,i_LSTM,i_MLP,i_RF,i_SVM,p_AutoGluon,p_GBT,p_KNN,p_LR,p_MLP,p_RF,p_SVM
c_AutoGluon,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,---,0.0875,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.5779,<0.0001***,<0.0001***,<0.0001***,0.3971,<0.0001***
c_GBT,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
c_KNN,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
c_LR,<0.0001***,<0.0001***,<0.0001***,---,0.4446,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.2579,<0.0001***,<0.0001***
c_MLP,<0.0001***,<0.0001***,<0.0001***,0.4446,---,---,<0.0001***,0.8059,<0.0001***,<0.0001***,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,0.0004***,<0.0001***,<0.0001***,<0.0001***,0.6473,<0.0001***,<0.0001***
c_MLP+LSTM,---,---,---,---,---,---,---,---,---,---,---,---,<0.0001***,---,---,---,---,---,---,---,---,---,---
c_RF,0.0875,<0.0001***,<0.0001***,<0.0001***,<0.0001***,---,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.4079,<0.0001***,<0.0001***,<0.0001***,0.3996,<0.0001***
c_SVM,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.8059,---,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,0.0002***,<0.0001***,<0.0001***,<0.0001***,0.7756,<0.0001***,<0.0001***
i_AutoGluon,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
i_GBT,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,---,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
